# Snowflake AppSec Inventory PoC Notebook

This notebook is a **sandbox proof of concept** for generating an AppSec-style report for Snowflake Python workloads.

It helps answer:

- Which Python UDFs / stored procedures exist?
- Which Python packages are requested or installed?
- Which objects use external access integrations or secrets?
- Which deployed handlers contain obvious risky patterns?
- Which packages match an optional vulnerability feed table?
- What can Snowflake-native controls cover, and where do you still need Snyk or an equivalent SDLC/AppSec toolchain?

## Important limits

This is **not** a full Snyk replacement.

It does **inventory + basic policy checks + regex-based code scanning**. It does **not** provide full SAST dataflow analysis, exploitability analysis, PR gates, IDE guidance, transitive dependency intelligence, or vendor-grade vulnerability intelligence.

Use this to support a CTO/QSA conversation, not as the final production control.

## 0. Expected permissions

Run this notebook with a role that has visibility into the sandbox objects you want to assess.

Recommended sandbox privileges:

- `USAGE` on target databases and schemas.
- Visibility into Python UDFs / stored procedures via `INFORMATION_SCHEMA`.
- Ability to run `GET_DDL()` for target objects where possible.
- `CREATE SCHEMA` or `CREATE TABLE` in the chosen report database/schema if you want persisted report tables.
- Optional: access to `SNOWFLAKE.ACCOUNT_USAGE` if you later extend this PoC account-wide.

For package policy testing, create/apply package policies only in a non-production sandbox first.

In [ ]:
# 1. Bootstrap Snowflake notebook session

from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.exceptions import SnowparkSQLException
import pandas as pd
import re
import json
import hashlib
from datetime import datetime, timezone

session = get_active_session()

pd.set_option("display.max_colwidth", 200)
pd.set_option("display.max_rows", 200)

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_UTC")

print("Connected Snowflake session")
print("User      :", session.get_current_user())
print("Role      :", session.get_current_role())
print("Database  :", session.get_current_database())
print("Schema    :", session.get_current_schema())
print("Warehouse :", session.get_current_warehouse())
print("Run ID    :", RUN_ID)

In [ ]:
# 2. Parameters - edit these for your sandbox

# Empty list = scan all accessible non-system databases returned by SHOW DATABASES.
# For safer first run, set this explicitly, for example:
# DATABASES_TO_SCAN = ["MY_SANDBOX_DB"]
DATABASES_TO_SCAN = []

# Skip system/shared/internal DBs by default.
DATABASE_EXCLUDE_REGEX = r"^(SNOWFLAKE|SNOWFLAKE_SAMPLE_DATA|UTIL_DB|UTILS|INFORMATION_SCHEMA)$"

# Optional schema filters. Leave empty to scan all schemas in selected DBs.
SCHEMAS_TO_INCLUDE = []     # Example: ["APP", "PUBLIC"]
SCHEMAS_TO_EXCLUDE = ["INFORMATION_SCHEMA"]

# Protect your sandbox from huge first runs.
MAX_OBJECTS_TO_GET_DDL = 500

# Create persisted report tables?
CREATE_REPORT_TABLES = True

# Report destination. Default: current database + APPSEC_POC schema.
current_db = session.get_current_database()
REPORT_DB = current_db.replace('"', '') if current_db else None
REPORT_SCHEMA = "APPSEC_POC"

# Optional vulnerability feed table.
# Expected columns, case-insensitive:
# PACKAGE_NAME, AFFECTED_SPEC, VULN_ID, SEVERITY, FIX_VERSION, SOURCE_URL, SUMMARY
#
# Example value:
# VULN_FEED_TABLE = "SECURITY_SANDBOX.APPSEC_POC.PYTHON_VULN_FEED"
VULN_FEED_TABLE = ""

# Optional approved/prohibited package lists for policy simulation.
# Keep names lowercase. Empty APPROVED_PACKAGES means "do not enforce allowlist".
APPROVED_PACKAGES = set([
    # "snowflake-snowpark-python",
    # "requests",
    # "pandas",
])

PROHIBITED_PACKAGES = set([
    "pycrypto",
    "pickle5",
])

# Minimum severity threshold only affects optional display filtering, not table generation.
SEVERITY_ORDER = {"INFO": 0, "LOW": 1, "MEDIUM": 2, "HIGH": 3, "CRITICAL": 4}

print("Configured report destination:", f"{REPORT_DB}.{REPORT_SCHEMA}" if REPORT_DB else "(no current DB)")

In [ ]:
# 3. Helper functions

def sql_quote_ident(name: str) -> str:
    """Double-quote a Snowflake identifier safely."""
    if name is None:
        raise ValueError("Identifier cannot be None")
    return '"' + str(name).replace('"', '""') + '"'

def sql_literal(value: str) -> str:
    """Single-quote a SQL string literal safely."""
    if value is None:
        return "NULL"
    return "'" + str(value).replace("'", "''") + "'"

def run_sql(sql: str, quiet: bool = False):
    if not quiet:
        print(sql)
    return session.sql(sql).collect()

def try_sql(sql: str, quiet: bool = True):
    try:
        return session.sql(sql).collect()
    except Exception as e:
        return {"error": str(e), "sql": sql}

def rows_to_pandas(rows):
    if isinstance(rows, dict) and "error" in rows:
        return pd.DataFrame([rows])
    return pd.DataFrame([r.as_dict() for r in rows])

def normalize_colname(c):
    return str(c).upper()

def get_df(sql: str) -> pd.DataFrame:
    return session.sql(sql).to_pandas()

def get_show_df(show_sql: str) -> pd.DataFrame:
    """Run a SHOW command and return result_scan as pandas."""
    session.sql(show_sql).collect()
    return session.sql("SELECT * FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()))").to_pandas()

def table_exists_fq(fq_table: str) -> bool:
    try:
        session.sql(f"SELECT 1 FROM {fq_table} LIMIT 1").collect()
        return True
    except Exception:
        return False

def safe_display(df, n=20, title=None):
    if title:
        print(f"\n=== {title} ===")
    if df is None:
        print("(None)")
    elif len(df) == 0:
        print("(no rows)")
    else:
        display(df.head(n))

In [ ]:
# 4. Discover databases and schemas

def discover_databases():
    if DATABASES_TO_SCAN:
        return DATABASES_TO_SCAN

    df = get_show_df("SHOW DATABASES")
    cols = {c.lower(): c for c in df.columns}
    name_col = cols.get("name") or cols.get("database_name") or list(df.columns)[0]

    dbs = []
    for name in df[name_col].dropna().astype(str).tolist():
        if re.match(DATABASE_EXCLUDE_REGEX, name, re.IGNORECASE):
            continue
        dbs.append(name)
    return dbs

def discover_schemas(db_name):
    try:
        df = get_df(f"""
            SELECT SCHEMA_NAME
            FROM {sql_quote_ident(db_name)}.INFORMATION_SCHEMA.SCHEMATA
            WHERE CATALOG_NAME = {sql_literal(db_name)}
            ORDER BY SCHEMA_NAME
        """)
    except Exception as e:
        print(f"Could not read schemas for {db_name}: {e}")
        return []

    schemas = []
    include_set = {s.upper() for s in SCHEMAS_TO_INCLUDE}
    exclude_set = {s.upper() for s in SCHEMAS_TO_EXCLUDE}

    for s in df["SCHEMA_NAME"].dropna().astype(str).tolist():
        if include_set and s.upper() not in include_set:
            continue
        if s.upper() in exclude_set:
            continue
        schemas.append(s)
    return schemas

databases = discover_databases()
print("Databases selected:", databases)

db_schema_pairs = []
for db in databases:
    schemas = discover_schemas(db)
    for s in schemas:
        db_schema_pairs.append((db, s))

print("Schema count selected:", len(db_schema_pairs))
safe_display(pd.DataFrame(db_schema_pairs, columns=["DATABASE", "SCHEMA"]), 50, "Selected database/schema pairs")

In [ ]:
# 5. Inventory Python UDFs and stored procedures from INFORMATION_SCHEMA

def get_view_columns(db_name, view_name):
    try:
        df = get_df(f"SELECT * FROM {sql_quote_ident(db_name)}.INFORMATION_SCHEMA.{view_name} LIMIT 0")
        return [normalize_colname(c) for c in df.columns]
    except Exception as e:
        print(f"Could not inspect {db_name}.INFORMATION_SCHEMA.{view_name}: {e}")
        return []

def select_expr(available_cols, col, alias=None):
    alias = alias or col
    if normalize_colname(col) in available_cols:
        return f"{sql_quote_ident(col)} AS {sql_quote_ident(alias)}"
    return f"NULL AS {sql_quote_ident(alias)}"

def inventory_functions_for_schema(db, schema):
    available = get_view_columns(db, "FUNCTIONS")
    if not available:
        return pd.DataFrame()

    select_sql = f"""
        SELECT
            'FUNCTION' AS OBJECT_TYPE,
            {select_expr(available, 'FUNCTION_CATALOG', 'OBJECT_CATALOG')},
            {select_expr(available, 'FUNCTION_SCHEMA', 'OBJECT_SCHEMA')},
            {select_expr(available, 'FUNCTION_NAME', 'OBJECT_NAME')},
            {select_expr(available, 'FUNCTION_OWNER', 'OBJECT_OWNER')},
            {select_expr(available, 'ARGUMENT_SIGNATURE', 'ARGUMENT_SIGNATURE')},
            {select_expr(available, 'DATA_TYPE', 'RETURN_TYPE')},
            {select_expr(available, 'FUNCTION_LANGUAGE', 'LANGUAGE')},
            {select_expr(available, 'FUNCTION_DEFINITION', 'OBJECT_DEFINITION')},
            {select_expr(available, 'PACKAGES', 'PACKAGES_REQUESTED')},
            {select_expr(available, 'INSTALLED_PACKAGES', 'INSTALLED_PACKAGES')},
            {select_expr(available, 'RUNTIME_VERSION', 'RUNTIME_VERSION')},
            {select_expr(available, 'CREATED', 'CREATED')},
            {select_expr(available, 'LAST_ALTERED', 'LAST_ALTERED')}
        FROM {sql_quote_ident(db)}.INFORMATION_SCHEMA.FUNCTIONS
        WHERE FUNCTION_SCHEMA = {sql_literal(schema)}
          AND UPPER(COALESCE(FUNCTION_LANGUAGE, '')) = 'PYTHON'
    """
    try:
        return get_df(select_sql)
    except Exception as e:
        print(f"Function inventory failed for {db}.{schema}: {e}")
        return pd.DataFrame()

def inventory_procedures_for_schema(db, schema):
    available = get_view_columns(db, "PROCEDURES")
    if not available:
        return pd.DataFrame()

    select_sql = f"""
        SELECT
            'PROCEDURE' AS OBJECT_TYPE,
            {select_expr(available, 'PROCEDURE_CATALOG', 'OBJECT_CATALOG')},
            {select_expr(available, 'PROCEDURE_SCHEMA', 'OBJECT_SCHEMA')},
            {select_expr(available, 'PROCEDURE_NAME', 'OBJECT_NAME')},
            {select_expr(available, 'PROCEDURE_OWNER', 'OBJECT_OWNER')},
            {select_expr(available, 'ARGUMENT_SIGNATURE', 'ARGUMENT_SIGNATURE')},
            {select_expr(available, 'DATA_TYPE', 'RETURN_TYPE')},
            {select_expr(available, 'PROCEDURE_LANGUAGE', 'LANGUAGE')},
            {select_expr(available, 'PROCEDURE_DEFINITION', 'OBJECT_DEFINITION')},
            {select_expr(available, 'PACKAGES', 'PACKAGES_REQUESTED')},
            {select_expr(available, 'INSTALLED_PACKAGES', 'INSTALLED_PACKAGES')},
            {select_expr(available, 'RUNTIME_VERSION', 'RUNTIME_VERSION')},
            {select_expr(available, 'CREATED', 'CREATED')},
            {select_expr(available, 'LAST_ALTERED', 'LAST_ALTERED')}
        FROM {sql_quote_ident(db)}.INFORMATION_SCHEMA.PROCEDURES
        WHERE PROCEDURE_SCHEMA = {sql_literal(schema)}
          AND UPPER(COALESCE(PROCEDURE_LANGUAGE, '')) = 'PYTHON'
    """
    try:
        return get_df(select_sql)
    except Exception as e:
        print(f"Procedure inventory failed for {db}.{schema}: {e}")
        return pd.DataFrame()

inventory_parts = []
for db, schema in db_schema_pairs:
    fdf = inventory_functions_for_schema(db, schema)
    pdf = inventory_procedures_for_schema(db, schema)
    if len(fdf):
        inventory_parts.append(fdf)
    if len(pdf):
        inventory_parts.append(pdf)

if inventory_parts:
    object_inventory = pd.concat(inventory_parts, ignore_index=True)
else:
    object_inventory = pd.DataFrame(columns=[
        "OBJECT_TYPE", "OBJECT_CATALOG", "OBJECT_SCHEMA", "OBJECT_NAME", "OBJECT_OWNER",
        "ARGUMENT_SIGNATURE", "RETURN_TYPE", "LANGUAGE", "OBJECT_DEFINITION",
        "PACKAGES_REQUESTED", "INSTALLED_PACKAGES", "RUNTIME_VERSION", "CREATED", "LAST_ALTERED"
    ])

def make_object_key(row):
    return f"{row.get('OBJECT_TYPE')}|{row.get('OBJECT_CATALOG')}.{row.get('OBJECT_SCHEMA')}.{row.get('OBJECT_NAME')}{row.get('ARGUMENT_SIGNATURE') or '()'}"

object_inventory["OBJECT_KEY"] = object_inventory.apply(make_object_key, axis=1)
object_inventory["RUN_ID"] = RUN_ID

print("Python object count:", len(object_inventory))
safe_display(object_inventory[[
    "OBJECT_TYPE", "OBJECT_CATALOG", "OBJECT_SCHEMA", "OBJECT_NAME",
    "ARGUMENT_SIGNATURE", "OBJECT_OWNER", "RUNTIME_VERSION", "PACKAGES_REQUESTED", "INSTALLED_PACKAGES"
]], 50, "Python object inventory")

In [ ]:
# 6. Best-effort GET_DDL for richer parsing

def build_get_ddl_object_name(row):
    db = row["OBJECT_CATALOG"]
    schema = row["OBJECT_SCHEMA"]
    name = row["OBJECT_NAME"]
    args = row.get("ARGUMENT_SIGNATURE") or "()"
    return f"{sql_quote_ident(db)}.{sql_quote_ident(schema)}.{sql_quote_ident(name)}{args}"

def get_object_ddl(row):
    obj_type = row["OBJECT_TYPE"]
    object_name = build_get_ddl_object_name(row)
    sql = f"SELECT GET_DDL({sql_literal(obj_type)}, {sql_literal(object_name)}) AS DDL"
    try:
        out = session.sql(sql).collect()
        return out[0]["DDL"], None
    except Exception as e:
        return None, str(e)

ddl_records = []
objects_to_scan = object_inventory.head(MAX_OBJECTS_TO_GET_DDL).copy()

for idx, row in objects_to_scan.iterrows():
    ddl, err = get_object_ddl(row)
    ddl_records.append({
        "RUN_ID": RUN_ID,
        "OBJECT_KEY": row["OBJECT_KEY"],
        "OBJECT_TYPE": row["OBJECT_TYPE"],
        "OBJECT_CATALOG": row["OBJECT_CATALOG"],
        "OBJECT_SCHEMA": row["OBJECT_SCHEMA"],
        "OBJECT_NAME": row["OBJECT_NAME"],
        "ARGUMENT_SIGNATURE": row.get("ARGUMENT_SIGNATURE"),
        "DDL": ddl,
        "DDL_SHA256": hashlib.sha256((ddl or "").encode("utf-8")).hexdigest() if ddl else None,
        "DDL_ERROR": err
    })

ddl_df = pd.DataFrame(ddl_records)
print("GET_DDL attempted:", len(ddl_df))
print("GET_DDL successful:", ddl_df["DDL"].notna().sum() if len(ddl_df) else 0)
safe_display(ddl_df[ddl_df["DDL_ERROR"].notna()][["OBJECT_TYPE", "OBJECT_CATALOG", "OBJECT_SCHEMA", "OBJECT_NAME", "DDL_ERROR"]], 20, "GET_DDL errors")

In [ ]:
# 7. Parse packages, external access integrations, and secrets from DDL / metadata

def extract_clause_parentheses(text, keyword):
    """
    Extract content inside KEYWORD = (...) from a DDL string.
    Handles basic nested parentheses and quoted strings.
    """
    if not text:
        return []
    pattern = re.compile(rf"\b{re.escape(keyword)}\b\s*=\s*\(", re.IGNORECASE)
    out = []
    for m in pattern.finditer(text):
        start = m.end()
        depth = 1
        i = start
        in_single = False
        in_double = False
        while i < len(text):
            ch = text[i]
            nxt = text[i+1] if i+1 < len(text) else ""
            if in_single:
                if ch == "'" and nxt == "'":
                    i += 2
                    continue
                elif ch == "'":
                    in_single = False
            elif in_double:
                if ch == '"':
                    in_double = False
            else:
                if ch == "'":
                    in_single = True
                elif ch == '"':
                    in_double = True
                elif ch == "(":
                    depth += 1
                elif ch == ")":
                    depth -= 1
                    if depth == 0:
                        out.append(text[start:i])
                        break
            i += 1
    return out

def split_csv_like(s):
    """Split comma-separated values while respecting simple quotes."""
    if not s:
        return []
    vals = []
    cur = []
    in_single = False
    in_double = False
    i = 0
    while i < len(s):
        ch = s[i]
        nxt = s[i+1] if i+1 < len(s) else ""
        if in_single:
            cur.append(ch)
            if ch == "'" and nxt == "'":
                cur.append(nxt)
                i += 2
                continue
            elif ch == "'":
                in_single = False
        elif in_double:
            cur.append(ch)
            if ch == '"':
                in_double = False
        else:
            if ch == "'":
                in_single = True
                cur.append(ch)
            elif ch == '"':
                in_double = True
                cur.append(ch)
            elif ch == ",":
                vals.append("".join(cur).strip())
                cur = []
            else:
                cur.append(ch)
        i += 1
    if cur:
        vals.append("".join(cur).strip())
    return [v for v in vals if v]

def strip_quotes(v):
    v = str(v).strip()
    if len(v) >= 2 and ((v[0] == "'" and v[-1] == "'") or (v[0] == '"' and v[-1] == '"')):
        v = v[1:-1]
    return v.replace("''", "'").strip()

def parse_package_spec(pkg_spec):
    """
    Best-effort parse of Python package spec:
      requests==2.31.0
      pandas
      urllib3>=1.26,<2
      snowflake-snowpark-python
    """
    raw = strip_quotes(pkg_spec)
    raw = raw.strip()
    m = re.match(r"^\s*([A-Za-z0-9_.\-]+)", raw)
    name = m.group(1).lower().replace("_", "-") if m else raw.lower()
    version = None
    pinned = False
    m2 = re.search(r"==\s*([A-Za-z0-9_.!\-+]+)", raw)
    if m2:
        version = m2.group(1)
        pinned = True
    return {
        "PACKAGE_SPEC": raw,
        "PACKAGE_NAME": name,
        "PINNED_VERSION": version,
        "IS_PINNED_EXACT": pinned
    }

def parse_packages_from_text(text):
    values = []
    for clause in extract_clause_parentheses(text, "PACKAGES"):
        values.extend(split_csv_like(clause))
    return [parse_package_spec(v) for v in values]

def parse_integrations_from_ddl(ddl):
    vals = []
    for clause in extract_clause_parentheses(ddl, "EXTERNAL_ACCESS_INTEGRATIONS"):
        vals.extend([strip_quotes(v) for v in split_csv_like(clause)])
    return [v for v in vals if v]

def parse_secrets_from_ddl(ddl):
    vals = []
    for clause in extract_clause_parentheses(ddl, "SECRETS"):
        parts = split_csv_like(clause)
        for p in parts:
            vals.append(strip_quotes(p))
    return [v for v in vals if v]

def parse_metadata_package_column(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return []
    s = str(value).strip()
    if not s:
        return []
    s2 = s.strip()
    if s2.startswith("[") and s2.endswith("]"):
        s2 = s2[1:-1]
    return [parse_package_spec(v) for v in split_csv_like(s2)]

package_rows = []
integration_rows = []
secret_rows = []

ddl_lookup = {}
if len(ddl_df):
    ddl_lookup = {r["OBJECT_KEY"]: r.get("DDL") for _, r in ddl_df.iterrows()}

for _, obj in object_inventory.iterrows():
    object_key = obj["OBJECT_KEY"]
    ddl = ddl_lookup.get(object_key) or ""
    packages = []

    packages.extend(parse_packages_from_text(ddl))
    packages.extend(parse_metadata_package_column(obj.get("PACKAGES_REQUESTED")))
    packages.extend(parse_metadata_package_column(obj.get("INSTALLED_PACKAGES")))

    seen = set()
    for p in packages:
        key = (p["PACKAGE_NAME"], p["PACKAGE_SPEC"])
        if key in seen:
            continue
        seen.add(key)
        package_rows.append({
            "RUN_ID": RUN_ID,
            "OBJECT_KEY": object_key,
            "OBJECT_TYPE": obj["OBJECT_TYPE"],
            "OBJECT_CATALOG": obj["OBJECT_CATALOG"],
            "OBJECT_SCHEMA": obj["OBJECT_SCHEMA"],
            "OBJECT_NAME": obj["OBJECT_NAME"],
            **p,
            "APPROVED_LIST_STATUS": (
                "NOT_EVALUATED" if not APPROVED_PACKAGES else
                "APPROVED" if p["PACKAGE_NAME"] in APPROVED_PACKAGES else
                "NOT_APPROVED"
            ),
            "PROHIBITED_STATUS": "PROHIBITED" if p["PACKAGE_NAME"] in PROHIBITED_PACKAGES else "OK"
        })

    for integ in parse_integrations_from_ddl(ddl):
        integration_rows.append({
            "RUN_ID": RUN_ID,
            "OBJECT_KEY": object_key,
            "OBJECT_TYPE": obj["OBJECT_TYPE"],
            "OBJECT_CATALOG": obj["OBJECT_CATALOG"],
            "OBJECT_SCHEMA": obj["OBJECT_SCHEMA"],
            "OBJECT_NAME": obj["OBJECT_NAME"],
            "EXTERNAL_ACCESS_INTEGRATION": integ
        })

    for sec in parse_secrets_from_ddl(ddl):
        secret_rows.append({
            "RUN_ID": RUN_ID,
            "OBJECT_KEY": object_key,
            "OBJECT_TYPE": obj["OBJECT_TYPE"],
            "OBJECT_CATALOG": obj["OBJECT_CATALOG"],
            "OBJECT_SCHEMA": obj["OBJECT_SCHEMA"],
            "OBJECT_NAME": obj["OBJECT_NAME"],
            "SECRET_REFERENCE_RAW": sec
        })

package_inventory = pd.DataFrame(package_rows)
integration_inventory = pd.DataFrame(integration_rows)
secret_inventory = pd.DataFrame(secret_rows)

safe_display(package_inventory, 50, "Package inventory")
safe_display(integration_inventory, 50, "External access integration usage")
safe_display(secret_inventory, 50, "Secret references parsed from DDL")

In [ ]:
# 8. Basic deployed-code pattern scanning
#
# This is intentionally regex-based and conservative. Treat results as triage signals,
# not as final SAST findings.

CODE_RULES = [
    {
        "RULE_ID": "SECRET_LITERAL_GENERIC",
        "SEVERITY": "HIGH",
        "CATEGORY": "Secret Exposure",
        "PATTERN": r"(?i)\b(api[_-]?key|secret|token|password|passwd|pwd)\b\s*=\s*['\"][^'\"]{8,}['\"]",
        "DESCRIPTION": "Potential hardcoded credential or token literal.",
        "REMEDIATION": "Move the value to a Snowflake SECRET object or enterprise secrets manager; rotate if exposed."
    },
    {
        "RULE_ID": "AWS_ACCESS_KEY_PATTERN",
        "SEVERITY": "CRITICAL",
        "CATEGORY": "Secret Exposure",
        "PATTERN": r"\bAKIA[0-9A-Z]{16}\b",
        "DESCRIPTION": "Potential AWS access key ID in deployed code.",
        "REMEDIATION": "Revoke/rotate the key and remove it from source history."
    },
    {
        "RULE_ID": "PRIVATE_KEY_BLOCK",
        "SEVERITY": "CRITICAL",
        "CATEGORY": "Secret Exposure",
        "PATTERN": r"-----BEGIN (RSA |EC |OPENSSH |DSA )?PRIVATE KEY-----",
        "DESCRIPTION": "Private key material appears in deployed code.",
        "REMEDIATION": "Remove private key, rotate credential, and use secrets management."
    },
    {
        "RULE_ID": "VERIFY_FALSE",
        "SEVERITY": "HIGH",
        "CATEGORY": "Transport Security",
        "PATTERN": r"(?is)\brequests\.(get|post|put|patch|delete|request)\s*\([^)]*verify\s*=\s*False",
        "DESCRIPTION": "TLS certificate validation appears disabled for outbound HTTP request.",
        "REMEDIATION": "Require certificate validation; do not use verify=False except in controlled test code."
    },
    {
        "RULE_ID": "PLAINTEXT_HTTP",
        "SEVERITY": "MEDIUM",
        "CATEGORY": "Transport Security",
        "PATTERN": r"['\"]http://[^'\"]+['\"]",
        "DESCRIPTION": "Plain HTTP URL literal found.",
        "REMEDIATION": "Use HTTPS endpoints and validate external access network rules."
    },
    {
        "RULE_ID": "EVAL_EXEC_USAGE",
        "SEVERITY": "HIGH",
        "CATEGORY": "Unsafe Code Execution",
        "PATTERN": r"\b(eval|exec)\s*\(",
        "DESCRIPTION": "Dynamic code execution detected.",
        "REMEDIATION": "Avoid eval/exec; replace with explicit parsing or dispatch logic."
    },
    {
        "RULE_ID": "PICKLE_LOADS",
        "SEVERITY": "HIGH",
        "CATEGORY": "Unsafe Deserialization",
        "PATTERN": r"\bpickle\.loads?\s*\(",
        "DESCRIPTION": "Pickle deserialization detected.",
        "REMEDIATION": "Avoid pickle for untrusted input; use safe serialization formats such as JSON."
    },
    {
        "RULE_ID": "YAML_LOAD_UNSAFE",
        "SEVERITY": "HIGH",
        "CATEGORY": "Unsafe Deserialization",
        "PATTERN": r"\byaml\.load\s*\(",
        "DESCRIPTION": "yaml.load detected; may be unsafe depending on Loader usage.",
        "REMEDIATION": "Use yaml.safe_load for untrusted data."
    },
    {
        "RULE_ID": "FSTRING_SESSION_SQL",
        "SEVERITY": "MEDIUM",
        "CATEGORY": "Injection Risk",
        "PATTERN": r"session\.sql\s*\(\s*f['\"]",
        "DESCRIPTION": "Potential dynamic SQL construction via f-string.",
        "REMEDIATION": "Use bind variables / parameters where possible; validate identifiers strictly."
    },
    {
        "RULE_ID": "STRING_CONCAT_SESSION_SQL",
        "SEVERITY": "MEDIUM",
        "CATEGORY": "Injection Risk",
        "PATTERN": r"session\.sql\s*\([^)]*\+[^)]*\)",
        "DESCRIPTION": "Potential dynamic SQL construction via string concatenation.",
        "REMEDIATION": "Use bind variables / parameters where possible; validate identifiers strictly."
    },
]

def get_scan_text_for_object(row):
    ddl = ddl_lookup.get(row["OBJECT_KEY"]) or ""
    definition = row.get("OBJECT_DEFINITION") or ""
    return ddl if ddl else definition

finding_rows = []

for _, obj in object_inventory.iterrows():
    text = get_scan_text_for_object(obj)
    if not text:
        continue

    for rule in CODE_RULES:
        pattern = re.compile(rule["PATTERN"], re.IGNORECASE | re.MULTILINE | re.DOTALL)
        for m in pattern.finditer(text):
            start = max(0, m.start() - 80)
            end = min(len(text), m.end() + 80)
            snippet = text[start:end].replace("\n", "\\n")
            if "SECRET" in rule["CATEGORY"].upper() or "KEY" in rule["RULE_ID"]:
                snippet = re.sub(r"(['\"])[^'\"]{8,}(['\"])", r"\1***REDACTED***\2", snippet)
                snippet = re.sub(r"AKIA[0-9A-Z]{16}", "AKIA****************", snippet)

            finding_rows.append({
                "RUN_ID": RUN_ID,
                "OBJECT_KEY": obj["OBJECT_KEY"],
                "OBJECT_TYPE": obj["OBJECT_TYPE"],
                "OBJECT_CATALOG": obj["OBJECT_CATALOG"],
                "OBJECT_SCHEMA": obj["OBJECT_SCHEMA"],
                "OBJECT_NAME": obj["OBJECT_NAME"],
                "RULE_ID": rule["RULE_ID"],
                "SEVERITY": rule["SEVERITY"],
                "CATEGORY": rule["CATEGORY"],
                "DESCRIPTION": rule["DESCRIPTION"],
                "REMEDIATION": rule["REMEDIATION"],
                "MATCH_SNIPPET_REDACTED": snippet,
            })

code_findings = pd.DataFrame(finding_rows)
safe_display(code_findings, 50, "Basic code findings")

In [ ]:
# 9. Package policy simulation findings

policy_rows = []

if len(package_inventory):
    for _, p in package_inventory.iterrows():
        if p.get("PROHIBITED_STATUS") == "PROHIBITED":
            policy_rows.append({
                "RUN_ID": RUN_ID,
                "OBJECT_KEY": p["OBJECT_KEY"],
                "OBJECT_TYPE": p["OBJECT_TYPE"],
                "OBJECT_CATALOG": p["OBJECT_CATALOG"],
                "OBJECT_SCHEMA": p["OBJECT_SCHEMA"],
                "OBJECT_NAME": p["OBJECT_NAME"],
                "PACKAGE_NAME": p["PACKAGE_NAME"],
                "PACKAGE_SPEC": p["PACKAGE_SPEC"],
                "RULE_ID": "PROHIBITED_PACKAGE",
                "SEVERITY": "HIGH",
                "DESCRIPTION": "Package is listed in PROHIBITED_PACKAGES.",
                "REMEDIATION": "Remove the package or document approved exception; enforce through Snowflake packages policy."
            })

        if APPROVED_PACKAGES and p.get("APPROVED_LIST_STATUS") == "NOT_APPROVED":
            policy_rows.append({
                "RUN_ID": RUN_ID,
                "OBJECT_KEY": p["OBJECT_KEY"],
                "OBJECT_TYPE": p["OBJECT_TYPE"],
                "OBJECT_CATALOG": p["OBJECT_CATALOG"],
                "OBJECT_SCHEMA": p["OBJECT_SCHEMA"],
                "OBJECT_NAME": p["OBJECT_NAME"],
                "PACKAGE_NAME": p["PACKAGE_NAME"],
                "PACKAGE_SPEC": p["PACKAGE_SPEC"],
                "RULE_ID": "PACKAGE_NOT_ON_APPROVED_LIST",
                "SEVERITY": "MEDIUM",
                "DESCRIPTION": "Package is not listed in APPROVED_PACKAGES.",
                "REMEDIATION": "Add to approved list after review or remove dependency."
            })

        if not bool(p.get("IS_PINNED_EXACT")):
            policy_rows.append({
                "RUN_ID": RUN_ID,
                "OBJECT_KEY": p["OBJECT_KEY"],
                "OBJECT_TYPE": p["OBJECT_TYPE"],
                "OBJECT_CATALOG": p["OBJECT_CATALOG"],
                "OBJECT_SCHEMA": p["OBJECT_SCHEMA"],
                "OBJECT_NAME": p["OBJECT_NAME"],
                "PACKAGE_NAME": p["PACKAGE_NAME"],
                "PACKAGE_SPEC": p["PACKAGE_SPEC"],
                "RULE_ID": "PACKAGE_VERSION_NOT_PINNED",
                "SEVERITY": "MEDIUM",
                "DESCRIPTION": "Package spec does not include an exact == pinned version.",
                "REMEDIATION": "Pin package versions and manage upgrades through CI/SCA workflow."
            })

package_policy_findings = pd.DataFrame(policy_rows)
safe_display(package_policy_findings, 50, "Package policy simulation findings")

In [ ]:
# 10. Optional vulnerability feed matching
#
# This expects a table with columns:
# PACKAGE_NAME, AFFECTED_SPEC, VULN_ID, SEVERITY, FIX_VERSION, SOURCE_URL, SUMMARY
#
# AFFECTED_SPEC examples:
#   <2.32.4
#   <=1.26.18
#   >=1.0,<2.0
#
# No feed table? This section returns an empty finding set.

try:
    from packaging.version import Version, InvalidVersion
except Exception:
    Version = None
    InvalidVersion = Exception

def parse_version(v):
    if v is None:
        return None
    s = str(v).strip()
    if not s:
        return None
    if Version:
        try:
            return Version(s)
        except Exception:
            return None
    nums = re.findall(r"\d+", s)
    return tuple(int(x) for x in nums) if nums else None

def compare_versions(installed, op, bound):
    vi = parse_version(installed)
    vb = parse_version(bound)
    if vi is None or vb is None:
        return False
    if op == "<":
        return vi < vb
    if op == "<=":
        return vi <= vb
    if op == ">":
        return vi > vb
    if op == ">=":
        return vi >= vb
    if op in ("=", "=="):
        return vi == vb
    return False

def affected_by_spec(installed_version, affected_spec):
    if installed_version is None or not affected_spec:
        return False
    clauses = [c.strip() for c in str(affected_spec).split(",") if c.strip()]
    if not clauses:
        return False
    for clause in clauses:
        m = re.match(r"^(<=|>=|<|>|==|=)\s*([A-Za-z0-9_.!\-+]+)$", clause)
        if not m:
            return False
        if not compare_versions(installed_version, m.group(1), m.group(2)):
            return False
    return True

def load_vuln_feed():
    if not VULN_FEED_TABLE:
        return pd.DataFrame(columns=[
            "PACKAGE_NAME", "AFFECTED_SPEC", "VULN_ID", "SEVERITY", "FIX_VERSION", "SOURCE_URL", "SUMMARY"
        ])
    df = get_df(f"SELECT * FROM {VULN_FEED_TABLE}")
    df.columns = [c.upper() for c in df.columns]
    for col in ["PACKAGE_NAME", "AFFECTED_SPEC", "VULN_ID", "SEVERITY", "FIX_VERSION", "SOURCE_URL", "SUMMARY"]:
        if col not in df.columns:
            df[col] = None
    df["PACKAGE_NAME"] = df["PACKAGE_NAME"].astype(str).str.lower().str.replace("_", "-", regex=False)
    df["SEVERITY"] = df["SEVERITY"].fillna("UNKNOWN").astype(str).str.upper()
    return df[["PACKAGE_NAME", "AFFECTED_SPEC", "VULN_ID", "SEVERITY", "FIX_VERSION", "SOURCE_URL", "SUMMARY"]]

vuln_feed = load_vuln_feed()
vuln_rows = []

if len(vuln_feed) and len(package_inventory):
    pkg_df = package_inventory.copy()
    pkg_df["PACKAGE_NAME_NORM"] = pkg_df["PACKAGE_NAME"].astype(str).str.lower().str.replace("_", "-", regex=False)

    for _, p in pkg_df.iterrows():
        matches = vuln_feed[vuln_feed["PACKAGE_NAME"] == p["PACKAGE_NAME_NORM"]]
        for _, v in matches.iterrows():
            if affected_by_spec(p.get("PINNED_VERSION"), v.get("AFFECTED_SPEC")):
                vuln_rows.append({
                    "RUN_ID": RUN_ID,
                    "OBJECT_KEY": p["OBJECT_KEY"],
                    "OBJECT_TYPE": p["OBJECT_TYPE"],
                    "OBJECT_CATALOG": p["OBJECT_CATALOG"],
                    "OBJECT_SCHEMA": p["OBJECT_SCHEMA"],
                    "OBJECT_NAME": p["OBJECT_NAME"],
                    "PACKAGE_NAME": p["PACKAGE_NAME"],
                    "PACKAGE_SPEC": p["PACKAGE_SPEC"],
                    "PINNED_VERSION": p.get("PINNED_VERSION"),
                    "VULN_ID": v.get("VULN_ID"),
                    "SEVERITY": v.get("SEVERITY"),
                    "AFFECTED_SPEC": v.get("AFFECTED_SPEC"),
                    "FIX_VERSION": v.get("FIX_VERSION"),
                    "SOURCE_URL": v.get("SOURCE_URL"),
                    "SUMMARY": v.get("SUMMARY"),
                    "REMEDIATION": f"Upgrade {p['PACKAGE_NAME']} to {v.get('FIX_VERSION') or 'a non-affected version'}; validate with SCA tool/official advisory."
                })

vuln_findings = pd.DataFrame(vuln_rows)
safe_display(vuln_feed, 20, "Loaded vulnerability feed")
safe_display(vuln_findings, 50, "Vulnerability matches")

In [ ]:
# 11. Best-effort integration detail lookup
#
# This tries DESCRIBE INTEGRATION for parsed external access integrations.
# Your role may not have privilege to view all details.

integration_detail_rows = []

if len(integration_inventory):
    for integ in sorted(integration_inventory["EXTERNAL_ACCESS_INTEGRATION"].dropna().astype(str).unique()):
        attempts = [
            f"DESC INTEGRATION {integ}",
            f"DESC INTEGRATION {sql_quote_ident(integ)}",
        ]
        success = False
        last_err = None
        for sql in attempts:
            try:
                session.sql(sql).collect()
                desc_df = session.sql("SELECT * FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()))").to_pandas()
                desc_df.columns = [c.upper() for c in desc_df.columns]
                for _, r in desc_df.iterrows():
                    integration_detail_rows.append({
                        "RUN_ID": RUN_ID,
                        "EXTERNAL_ACCESS_INTEGRATION": integ,
                        "PROPERTY": r.get("PROPERTY"),
                        "VALUE": r.get("VALUE"),
                        "DEFAULT": r.get("DEFAULT"),
                        "DESCRIPTION": r.get("DESCRIPTION")
                    })
                success = True
                break
            except Exception as e:
                last_err = str(e)
        if not success:
            integration_detail_rows.append({
                "RUN_ID": RUN_ID,
                "EXTERNAL_ACCESS_INTEGRATION": integ,
                "PROPERTY": "ERROR",
                "VALUE": last_err,
                "DEFAULT": None,
                "DESCRIPTION": "Could not describe integration with current role."
            })

integration_details = pd.DataFrame(integration_detail_rows)
safe_display(integration_details, 100, "External access integration details")

In [ ]:
# 12. Create executive summary tables in pandas

def count_by_severity(df, severity_col="SEVERITY"):
    if df is None or len(df) == 0 or severity_col not in df.columns:
        return {}
    return df[severity_col].fillna("UNKNOWN").astype(str).str.upper().value_counts().to_dict()

summary = {
    "RUN_ID": RUN_ID,
    "SCANNED_DATABASES": len(databases),
    "SCANNED_SCHEMAS": len(db_schema_pairs),
    "PYTHON_OBJECTS": len(object_inventory),
    "GET_DDL_SUCCESS": int(ddl_df["DDL"].notna().sum()) if len(ddl_df) else 0,
    "PACKAGE_REFERENCES": len(package_inventory),
    "UNIQUE_PACKAGES": int(package_inventory["PACKAGE_NAME"].nunique()) if len(package_inventory) else 0,
    "OBJECTS_WITH_EXTERNAL_ACCESS": int(integration_inventory["OBJECT_KEY"].nunique()) if len(integration_inventory) else 0,
    "OBJECTS_WITH_SECRET_REFERENCES": int(secret_inventory["OBJECT_KEY"].nunique()) if len(secret_inventory) else 0,
    "CODE_FINDINGS": len(code_findings),
    "PACKAGE_POLICY_FINDINGS": len(package_policy_findings),
    "VULN_FINDINGS": len(vuln_findings),
    "CODE_FINDINGS_BY_SEVERITY": json.dumps(count_by_severity(code_findings)),
    "PACKAGE_POLICY_FINDINGS_BY_SEVERITY": json.dumps(count_by_severity(package_policy_findings)),
    "VULN_FINDINGS_BY_SEVERITY": json.dumps(count_by_severity(vuln_findings)),
    "CREATED_AT_UTC": datetime.now(timezone.utc).isoformat()
}

summary_df = pd.DataFrame([summary])

risk_frames = []

if len(code_findings):
    tmp = code_findings[["RUN_ID", "OBJECT_KEY", "OBJECT_TYPE", "OBJECT_CATALOG", "OBJECT_SCHEMA", "OBJECT_NAME", "SEVERITY", "RULE_ID", "CATEGORY", "DESCRIPTION", "REMEDIATION"]].copy()
    tmp["FINDING_SOURCE"] = "CODE_REGEX"
    risk_frames.append(tmp)

if len(package_policy_findings):
    tmp = package_policy_findings[["RUN_ID", "OBJECT_KEY", "OBJECT_TYPE", "OBJECT_CATALOG", "OBJECT_SCHEMA", "OBJECT_NAME", "SEVERITY", "RULE_ID", "DESCRIPTION", "REMEDIATION"]].copy()
    tmp["CATEGORY"] = "Package Governance"
    tmp["FINDING_SOURCE"] = "PACKAGE_POLICY_SIMULATION"
    risk_frames.append(tmp)

if len(vuln_findings):
    tmp = vuln_findings[["RUN_ID", "OBJECT_KEY", "OBJECT_TYPE", "OBJECT_CATALOG", "OBJECT_SCHEMA", "OBJECT_NAME", "SEVERITY", "VULN_ID", "SUMMARY", "REMEDIATION"]].copy()
    tmp["RULE_ID"] = tmp["VULN_ID"]
    tmp["CATEGORY"] = "Dependency Vulnerability"
    tmp["DESCRIPTION"] = tmp["SUMMARY"]
    tmp["FINDING_SOURCE"] = "VULN_FEED_MATCH"
    tmp = tmp[["RUN_ID", "OBJECT_KEY", "OBJECT_TYPE", "OBJECT_CATALOG", "OBJECT_SCHEMA", "OBJECT_NAME", "SEVERITY", "RULE_ID", "CATEGORY", "DESCRIPTION", "REMEDIATION", "FINDING_SOURCE"]]
    risk_frames.append(tmp)

all_findings = pd.concat(risk_frames, ignore_index=True) if risk_frames else pd.DataFrame(columns=[
    "RUN_ID", "OBJECT_KEY", "OBJECT_TYPE", "OBJECT_CATALOG", "OBJECT_SCHEMA", "OBJECT_NAME",
    "SEVERITY", "RULE_ID", "CATEGORY", "DESCRIPTION", "REMEDIATION", "FINDING_SOURCE"
])

safe_display(summary_df, 5, "Executive summary")
safe_display(all_findings.sort_values(by=["SEVERITY", "OBJECT_CATALOG", "OBJECT_SCHEMA", "OBJECT_NAME"], ascending=False), 100, "All findings")

In [ ]:
# 13. Persist report tables to Snowflake

def ensure_report_schema():
    if not REPORT_DB:
        raise ValueError("REPORT_DB is empty. Set a report database before persisting tables.")
    session.sql(f"CREATE SCHEMA IF NOT EXISTS {sql_quote_ident(REPORT_DB)}.{sql_quote_ident(REPORT_SCHEMA)}").collect()

def write_report_table(df, table_name):
    if df is None:
        df = pd.DataFrame()
    if len(df) == 0:
        df = pd.DataFrame([{"RUN_ID": RUN_ID, "NOTE": "No rows generated for this table."}])
    session.write_pandas(
        df,
        table_name=table_name,
        database=REPORT_DB,
        schema=REPORT_SCHEMA,
        auto_create_table=True,
        overwrite=True,
        quote_identifiers=True
    )
    print(f"Wrote {REPORT_DB}.{REPORT_SCHEMA}.{table_name}: {len(df)} rows")

if CREATE_REPORT_TABLES:
    ensure_report_schema()

    write_report_table(summary_df, "APPSEC_POC_SUMMARY")
    write_report_table(object_inventory, "APPSEC_POC_OBJECT_INVENTORY")
    write_report_table(ddl_df.drop(columns=["DDL"], errors="ignore"), "APPSEC_POC_OBJECT_DDL_METADATA")
    write_report_table(package_inventory, "APPSEC_POC_PACKAGE_INVENTORY")
    write_report_table(integration_inventory, "APPSEC_POC_EXTERNAL_ACCESS_USAGE")
    write_report_table(integration_details, "APPSEC_POC_EXTERNAL_ACCESS_DETAILS")
    write_report_table(secret_inventory, "APPSEC_POC_SECRET_REFERENCES")
    write_report_table(code_findings, "APPSEC_POC_CODE_FINDINGS")
    write_report_table(package_policy_findings, "APPSEC_POC_PACKAGE_POLICY_FINDINGS")
    write_report_table(vuln_findings, "APPSEC_POC_VULN_FINDINGS")
    write_report_table(all_findings, "APPSEC_POC_ALL_FINDINGS")

    print("\nReport tables created.")
else:
    print("CREATE_REPORT_TABLES is False. Skipping persistence.")

In [ ]:
# 14. Optional: create an executive report view

if CREATE_REPORT_TABLES:
    fq_schema = f"{sql_quote_ident(REPORT_DB)}.{sql_quote_ident(REPORT_SCHEMA)}"
    view_sql = f"""
    CREATE OR REPLACE VIEW {fq_schema}."APPSEC_POC_EXECUTIVE_REPORT" AS
    SELECT
        f."RUN_ID",
        f."OBJECT_CATALOG",
        f."OBJECT_SCHEMA",
        f."OBJECT_NAME",
        f."OBJECT_TYPE",
        f."SEVERITY",
        f."FINDING_SOURCE",
        f."CATEGORY",
        f."RULE_ID",
        f."DESCRIPTION",
        f."REMEDIATION"
    FROM {fq_schema}."APPSEC_POC_ALL_FINDINGS" f
    WHERE COALESCE(f."NOTE", '') = ''
    ORDER BY
        CASE UPPER(f."SEVERITY")
            WHEN 'CRITICAL' THEN 5
            WHEN 'HIGH' THEN 4
            WHEN 'MEDIUM' THEN 3
            WHEN 'LOW' THEN 2
            WHEN 'INFO' THEN 1
            ELSE 0
        END DESC,
        f."OBJECT_CATALOG", f."OBJECT_SCHEMA", f."OBJECT_NAME"
    """
    session.sql(view_sql).collect()
    print(f"Created view: {REPORT_DB}.{REPORT_SCHEMA}.APPSEC_POC_EXECUTIVE_REPORT")

    display(get_df(f"SELECT * FROM {fq_schema}.\"APPSEC_POC_EXECUTIVE_REPORT\" LIMIT 100"))

## 15. Example package policy SQL for sandbox testing

Use this only after reviewing the package inventory. This is **example SQL**, not something this notebook automatically applies.

```sql
CREATE OR REPLACE PACKAGES POLICY SECURITY_SANDBOX.APPSEC_POC.PYTHON_PACKAGES_POLICY
  LANGUAGE PYTHON
  ALLOWLIST = ('snowflake-snowpark-python', 'requests', 'pandas')
  BLOCKLIST = ('pycrypto', 'pickle5')
  ADDITIONAL_CREATION_BLOCKLIST = ('pycrypto', 'pickle5');

ALTER ACCOUNT SET PACKAGES POLICY SECURITY_SANDBOX.APPSEC_POC.PYTHON_PACKAGES_POLICY;
```

In production, do not use a tiny allowlist without testing dependency resolution first. A strict allowlist can break stored procedures/UDF creation if transitive dependencies are not allowed.

In [ ]:
# 16. Optional helper: package policy candidate output
#
# This prints suggested allowlist/blocklist candidates based on observed packages.
# Review manually before using.

if len(package_inventory):
    observed = sorted(package_inventory["PACKAGE_NAME"].dropna().astype(str).str.lower().unique().tolist())
    prohibited_observed = sorted(set(observed).intersection(PROHIBITED_PACKAGES))

    print("Observed package names:")
    print(", ".join(observed))

    print("\nObserved prohibited package candidates:")
    print(", ".join(prohibited_observed) if prohibited_observed else "(none)")

    allow_sql = ", ".join([sql_literal(x) for x in observed]) if observed else ""
    block_sql = ", ".join([sql_literal(x) for x in sorted(PROHIBITED_PACKAGES)]) if PROHIBITED_PACKAGES else ""

    print("\nDraft package policy SQL - review before use:")
    print(f"""
CREATE OR REPLACE PACKAGES POLICY {sql_quote_ident(REPORT_DB)}.{sql_quote_ident(REPORT_SCHEMA)}."PYTHON_PACKAGES_POLICY_DRAFT"
  LANGUAGE PYTHON
  ALLOWLIST = ({allow_sql})
  BLOCKLIST = ({block_sql})
  ADDITIONAL_CREATION_BLOCKLIST = ({block_sql});
""")
else:
    print("No packages observed.")

## 17. What this PoC proves vs. does not prove

### This PoC can prove

- You can inventory deployed Snowflake Python objects.
- You can parse package usage and external access usage from Snowflake metadata/DDL.
- You can simulate package policy governance.
- You can generate report tables for QSA / CTO discussion.
- You can show why Snowflake-native controls are strong for runtime/data-plane governance.

### This PoC does not prove

- That Snowflake native tools can replace Snyk Code or full SAST.
- That Snowflake can perform production-grade dependency vulnerability intelligence by itself.
- That Git history / PR / IDE scanning is covered.
- That IaC or container image scanning is covered.
- That remediation workflow, ownership, SLAs, and PR gates are solved.

### Recommended follow-up

Run the same repo through one Snyk scan and this notebook report side-by-side:

- Findings unique to Snyk = SDLC/AppSec gap.
- Findings unique to Snowflake = runtime/data-plane governance gap.
- Overlap = good candidates for automated evidence reporting.